# Step 2: Remove predictors that repeat each other (collinearity, using F-tests)

Part of the model-selection steps in this folder (run in order the first time: step1 to step7). Shared setup (data,
random 80/20 train/test split by whole days, predictors, helpers) is in `model_utils.py`. The full explanation of
every step is in `docs/model_selection.md`.

**Needs:** `data/processed/model_table.parquet`. **Saves:** `data/processed/model_selection/step2_terms.json`
(the predictors kept), used by Steps 3 to 6.

**Collinearity** means two or more predictors carry the same information. The model then can't separate their effects,
so their coefficients become unreliable. Checks:

1. **Correlation** between the numeric predictors.
2. **How much of each predictor the others explain** (regress it on all other predictors and take the R²). Above
   0.8 = **flagged** as collinear.
3. **F-tests:**
   - For strongly correlated pairs (correlation above 0.7): if the pair is significant *together* but not *one at a
     time*, they share one piece of information, and the weaker one is dropped.
   - For each flagged predictor: if its own F-test is not significant, the information is already carried by the
     other predictors, so it is dropped.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import model_utils as mu

train, test = mu.load()

In [ ]:
corr = train[mu.CONTINUOUS].corr()
corr.round(2)

In [ ]:
rows = []
for v in mu.CONTINUOUS:
    others = [c for c in mu.CONTINUOUS if c != v]
    rows.append({"predictor": v,
                 "R2 explained by calendar + zone": mu.ols(mu.CALENDAR + ["zone"], target=v).rsquared,
                 "R2 explained by all other predictors": mu.ols(mu.CALENDAR + ["zone"] + others, target=v).rsquared})
aux = pd.DataFrame(rows).set_index("predictor")
aux["flagged"] = aux["R2 explained by all other predictors"] > mu.AUX_R2_FLAG
aux

In [ ]:
res_all = mu.ols(mu.ALL_TERMS, clustered=True)
single = mu.term_tests(res_all)

drop, decisions = set(), []
pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
for (a, b), r in pairs[pairs.abs() > mu.PAIR_CORR_FLAG].items():
    joint = float(res_all.f_test(f"{a} = 0, {b} = 0").pvalue)
    pa, pb = single.loc[a, "p"], single.loc[b, "p"]
    collinear = joint < mu.ALPHA and max(pa, pb) >= mu.ALPHA
    weaker = a if pa >= pb else b
    if collinear:
        drop.add(weaker)
    decisions.append({"check": f"pair {a} / {b} (correlation {r:.2f})", "p together": f"{joint:.4f}",
                      "p one at a time": f"{pa:.4f} / {pb:.4f}",
                      "decision": f"drop {weaker}" if collinear else "keep both (each significant on its own)"})
for v in aux.index[aux["flagged"]]:
    p = single.loc[v, "p"]
    if p >= mu.ALPHA:
        drop.add(v)
    decisions.append({"check": f"flagged {v} (R² {aux.loc[v, 'R2 explained by all other predictors']:.2f})",
                      "p one at a time": f"{p:.4f}",
                      "decision": f"drop {v}" if p >= mu.ALPHA else "keep (adds information of its own)"})
del res_all

step2_terms = mu.with_hierarchy([t for t in mu.ALL_TERMS if t not in drop])
mu.save("step2_terms", step2_terms)
print("dropped for collinearity:", sorted(set(mu.ALL_TERMS) - set(step2_terms)) or "none")
print("kept (saved for the next steps):", ", ".join(step2_terms))
pd.DataFrame(decisions).set_index("check").fillna("")

**Results.**
- **The two lags** are strongly correlated (0.88), but each is significant on its own, so **both are kept**.
- **Five predictors are flagged** (more than 80% explained by the others): temperature, sunlight, `is_day` and both
  lags.
- **`temperature_2m` and `is_day` are dropped.** About 86% of temperature and 82% of `is_day` is explained by the
  calendar alone: season and time of day. Neither adds significant information beyond that (p = 0.54 and 0.14).
  `temp_sq` goes with temperature.
- **Sunlight (p = 0.03) and both lags are kept,** because each adds information of its own.

Note: dropping temperature only means it doesn't help *predict* demand once the calendar is known. Temperature's
effect on demand is studied separately in the FWL section of `05_statistical_tests.ipynb`.